# Shuttle Classification

58,000 space-shuttle radiator and engine readings, each described by 7 sensor values. Predict the shuttle's state — a famously imbalanced dataset where one class holds almost 80% of all examples and three classes have fewer than 200 examples between them.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Statlog (Shuttle) (id 148)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/148/data.csv')))
print(df.shape)
print('Readings per class:')
print(df['class'].value_counts().sort_index())
df.head(3)


(58000, 8)
Readings per class:
class
1    45586
2       50
3      171
4     8903
5     3267
6       10
7       13
Name: count, dtype: int64


,,Rad Flow,Fpv Close,Fpv Open,High,Bypass,Bpv Close,Bpv Open,class
50,21,77,0,28,0,27,48,22,2
55,0,92,0,0,26,36,92,56,4
53,0,82,0,52,-5,29,30,2,1


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9986
Logistic Regression accuracy: 0.9657


              precision    recall  f1-score   support

           1       1.00      1.00      1.00     11396
           2       1.00      1.00      1.00        12
           3       0.89      0.74      0.81        43
           4       1.00      1.00      1.00      2226
           5       1.00      1.00      1.00       817
           6       0.00      0.00      0.00         3
           7       0.50      0.33      0.40         3

    accuracy                           1.00     14500
   macro avg       0.77      0.73      0.74     14500
weighted avg       1.00      1.00      1.00     14500



/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/hatch/workspace/github-batch11-work/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Which shuttle sensors matter most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Bypass       0.359597
Bpv Open     0.254927
Fpv Open     0.155222
Rad Flow     0.129602
Bpv Close    0.095284
High         0.003303
Fpv Close    0.002064
dtype: float64


Winner: Random Forest
